<a href="https://colab.research.google.com/github/serahnjogu-new/AI4EAC-Climate-Health-Risk-Prediction-Challenge-/blob/main/Winning_Climate_(0_838).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# 1. LOAD
train   = pd.read_csv('/content/Train.csv')
test    = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission.csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. FEATURE ENGINEERING — age and year focused
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0,
        3:1,  4:1, 5:1,
        6:2,  7:2, 8:2,
        9:3, 10:3, 11:3
    })

    # ── AGE FEATURES (most important!) ──
    df['log_age']      = np.log1p(df['age'])
    df['sqrt_age']     = np.sqrt(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_older_child']= (df['age'].between(5, 15)).astype(int)
    df['is_adult']     = (df['age'].between(15, 60)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['is_young']     = (df['age'] <= 5).astype(int)
    df['age_group']    = pd.cut(df['age'],
                                bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # ── YEAR FEATURES (second most important!) ──
    df['year_normalized'] = (df['year'] - 2007) / (2022 - 2007)
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769,
        2011: 0.658, 2012: 0.703, 2013: 0.563, 2014: 0.688,
        2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['is_early_period']  = (df['year'] <= 2010).astype(int)
    df['is_mid_period']    = (df['year'].between(2011, 2016)).astype(int)
    df['is_recent_period'] = (df['year'] >= 2017).astype(int)

    # ── AGE x YEAR INTERACTIONS (the magic!) ──
    df['year_x_age']        = df['year_normalized'] * df['age']
    df['year_x_log_age']    = df['year_normalized'] * df['log_age']
    df['year_sens_x_age']   = df['year_sensitivity'] * df['age']
    df['year_sens_x_infant']= df['year_sensitivity'] * df['is_infant']
    df['year_sens_x_child'] = df['year_sensitivity'] * df['is_child']
    df['early_infant']      = ((df['year'] <= 2010) & (df['age'] <= 1)).astype(int)
    df['early_child']       = ((df['year'] <= 2010) & (df['age'] <= 5)).astype(int)
    df['recent_elderly']    = ((df['year'] >= 2017) & (df['age'] >= 60)).astype(int)
    df['recent_infant']     = ((df['year'] >= 2017) & (df['age'] <= 1)).astype(int)
    df['mid_child']         = ((df['year'].between(2011,2016)) & (df['age'] <= 5)).astype(int)

    # ── ONLY THE BEST CLIMATE FEATURES ──
    df['temp_diff_30d']  = df['tmax_30d'] - df['tmin_30d']
    df['rain_intensity'] = df['rain_sum_30d'] / (df['rain_days_30d'] + 1)
    df['temp_range']     = df['max_temperature'] - df['min_temperature']

    # ── CATEGORICAL ──
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    return df.drop(['ID', 'deathdate', 'location', 'deathdate_extra'], axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X      = train_df.drop('is_climate_sensitive', axis=1)
y      = train_df['is_climate_sensitive']
X_test = test_df.copy()

X      = X.fillna(X.median())
X_test = X_test.fillna(X_test.median())

print("X shape:", X.shape)

# 3. OOF TRAINING
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg = (y==0).sum(); pos = (y==1).sum()

oof_lgbm  = np.zeros(len(X))
oof_xgb   = np.zeros(len(X))
test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))

print("\nTraining LGBM...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        scale_pos_weight=2.5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        importance_type='gain'
    )
    model.fit(X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(100), log_evaluation(0)]
    )
    oof_lgbm[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_lgbm         += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

print("\nTraining XGB...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        scale_pos_weight=neg/pos,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        random_state=42,
        verbosity=0
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_xgb         += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

# 4. ENSEMBLE
oof_ensemble  = (oof_lgbm + oof_xgb) / 2
test_ensemble = (test_lgbm + test_xgb) / 2

# 5. EVALUATE
print("\n=== RESULTS ===")
for name, oof in [('LGBM', oof_lgbm), ('XGB', oof_xgb), ('Ensemble', oof_ensemble)]:
    f1  = f1_score(y, (oof >= 0.5).astype(int))
    auc = roc_auc_score(y, oof)
    print(f"{name:10}: F1={f1:.4f}  AUC={auc:.4f}  Score={0.6*f1+0.4*auc:.4f}")

# 6. SUBMISSION
submission = pd.DataFrame({
    'ID':         ss['ID'],
    'TargetF1':   (test_ensemble >= 0.5).astype(int),
    'TargetRAUC': test_ensemble
})

submission.to_csv('submission_ageyear.csv', index=False)
print("\nTargetF1 distribution:")
print(submission['TargetF1'].value_counts())

from google.colab import files
files.download('submission_ageyear.csv')

X shape: (3146, 57)

Training LGBM...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001558 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6298
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 56
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>